In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
plt.rcParams['figure.dpi'] = 120

train = pd.read_parquet('../data/train_merged.parquet')
print(f'Loaded: {train.shape[0]:,} rows × {train.shape[1]} cols')
print(f'Fraud rate: {train["isFraud"].mean():.2%}')

In [ ]:
sample = train.sample(100000, random_state=42)
print(f'Sampled: {sample.shape[0]:,} rows × {sample.shape[1]} cols')
print(f'Sample fraud rate: {sample["isFraud"].mean():.2%}')
print(sample.head())
print(sample.columns.tolist())

Sampled: 100,000 rows × 434 cols
Sample fraud rate: 3.58%
        TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  \
470624        3457624        0       12153579      724.000000         W   
565820        3552820        0       15005886      108.500000         W   
284083        3271083        0        6970178       47.950001         W   
239689        3226689        0        5673658      100.598999         C   
281855        3268855        0        6886780      107.949997         W   

        card1  card2  card3       card4  card5  ... id_31  id_32  id_33  \
470624   7826  481.0  150.0  mastercard  224.0  ...  None    NaN   None   
565820  12544  321.0  150.0        visa  226.0  ...  None    NaN   None   
284083   9400  111.0  150.0  mastercard  224.0  ...  None    NaN   None   
239689  15885  545.0  185.0        visa  138.0  ...  None    NaN   None   
281855  15497  490.0  150.0        visa  226.0  ...  None    NaN   None   

        id_34  id_35 id_36 id_37  id_38 

### 📊 IEEE-CIS Feature Overview & Logic
Das Dataset ist in logische Gruppen unterteilt. Für das **Feature Engineering** und die **Controlling-Brille** sind folgende Kategorien entscheidend:

| Kategorie | Spalten | Beschreibung | Controlling-Relevanz |
| :--- | :--- | :--- | :--- |
| **Identity/Target** | `TransactionID`, `isFraud` | Eindeutige ID und Zielvariable. | Die Basis für die Berechnung der Fraud-Kosten. |
| **Transaction Info** | `TransactionDT`, `TransactionAmt` | Zeitstempel (Delta in Sek.) und Betrag in USD. | `TransactionAmt` bestimmt die Höhe des direkten Schadens (False Negatives). |
| **Product & Region** | `ProductCD`, `addr1`, `addr2`, `dist1/2` | Produkt-Typ, Adress-Regionen und Distanzen. | Geographische Anomalien (z.B. Billing != Shipping). |
| **Card Info** | `card1` - `card6` | Zahlungsmittel-Infos (Typ, Bank ID, Anbieter). | Identifikation von User-Clustern (UIDs). |
| **Email** | `P_emaildomain`, `R_emaildomain` | Maildomain von Käufer (Purchaser) & Empfänger. | Prüfung auf "Trash-Emails" oder riskante Provider. |
| **Counts (C)** | `C1` - `C14` | Anonymisierte Zählwerte (z.B. IP-Adress-Counts). | **Velocity-Check:** Wie oft taucht ein Merkmal aktuell auf? |
| **Timedeltas (D)** | `D1` - `D15` | Zeitliche Differenzen (z.B. Tage seit letztem Kauf). | Identifikation von "Account Takeover" oder neuen Usern. |
| **Matches (M)** | `M1` - `M9` | Abgleiche (True/False) z.B. Name vs. Karte. | Vertrauenswürdigkeit der Transaktions-Metadaten. |
| **Vesta (V)** | `V1` - `V339` | Vesta-spezifische Engineering-Features (Rankings etc.). | Hochdimensionaler Input für UMAP / PCA. |
| **Identity (id)** | `id_01` - `id_38` | Geräte-Infos, Browser, OS, Proxy-Status. | Erkennung von Bot-Netzwerken und Device-Fingerprinting. |

---
**💡 Strategische Notiz:** Die `V`-Spalten korrelieren oft stark. Im nächsten Schritt sollten wir diese mittels **PCA** oder **UMAP** (wie im Plan unter Punkt 1 vorgesehen) reduzieren, um das Modell nicht durch Multikollinearität zu verwässern.

In [ ]:
print(sample.describe())